# Energy Performance Certificates (DPE) vs Real Electricity Consumption Analysis

**Enedis Challenge - Open Data University**

## Objectives:
1. Compare DPE estimated consumption vs real measured consumption
2. Quantify savings when improving energy class (G→F, F→E, etc.)
3. Analyze variability due to individual behaviors

## Data:
- Simulated Enedis consumption data
- Simulated DPE data (based on ADEME methodology)
- Analysis limited to addresses with ≥10 dwellings

**Author:** Belén Vera Maric
**Date:** 05/06/2026

# Data Preparation: Merging DPE and ENEDIS Electricity Consumption Data

## Objective
Prepare a clean, merged dataset to analyze:
1. Energy savings when improving DPE class (G→F, F→E, ..., B→A)
2. Accuracy of DPE estimates compared to real electricity consumption

## Data Sources

### DPE Data (Diagnostic de Performance Énergétique)
- **Source**: data.gouv.fr - DPE logements
- **Contains BAN columns**: The French government pre-processed all DPE addresses using the **Base Adresse Nationale (BAN)** API
- **Key BAN columns**: `adresse_ban`, `code_insee_ban`, `nom_commune_ban`, coordinates

### ENEDIS Data (Electricity Consumption)
- **Source**: data.gouv.fr - Consommation électrique annuelle par adresse
- **Format**: CSV with semicolon separator
- **Key columns**: Address, INSEE code, consumption in MWh

## Why the Base Adresse Nationale (BAN)?

The BAN is France's official address database. It provides:
1. **Address normalization**: Converts differently formatted addresses to a standard format
2. **Geocoding**: Adds X/Y coordinates to each address
3. **Official codes**: Ensures consistent INSEE codes across datasets

**In this analysis**: The DPE data already includes BAN columns (pre-processed by the government). We use these to create a standardized join key. For ENEDIS, we apply the same normalization logic manually.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported")
print(f"Pandas version: {pd.__version__}")

Libraries imported
Pandas version: 2.3.3


## 1. Load DPE Data (Already Geocoded with BAN)

The DPE file comes from data.gouv.fr and **already contains BAN columns**. These columns are the result of calling the BAN API to normalize and geocode each address.

### Required DPE Columns (based on official schema `dpe03existant-tableschema.json`)

| Column | Description |
|--------|-------------|
| `adresse_ban` | Normalized address (BAN) |
| `code_insee_ban` | Official INSEE code (5 digits) |
| `nom_commune_ban` | Municipality name |
| `coordonnee_cartographique_x_ban` | X coordinate (geocoding) |
| `coordonnee_cartographique_y_ban` | Y coordinate (geocoding) |
| `etiquette_dpe` | DPE class (A through G) |
| `conso_5_usages_par_m2_ef` | Theoretical consumption (kWh/m²) |
| `surface_habitable_logement` | Livable area (m²) |
| `date_etablissement_dpe` | DPE establishment date |
| `type_batiment` | Building type (apartment/house) |

In [2]:
# Load DPE data with BAN columns
# ============================================
# NOTE: # The actual DPE file is ~2-3GB, so we filter by date during load
# ENEDIS data ranges from 2021-2024, so we keep matching DPE years - date_etablissement_dpe UNTIL 31/12/2024
# ============================================

# Define columns needed (optimize memory)
dpe_columns = [
    'adresse_ban', 'code_insee_ban', 'nom_commune_ban',
    'coordonnee_cartographique_x_ban', 'coordonnee_cartographique_y_ban',
    'etiquette_dpe', 'conso_5_usages_ef', 'conso_5_usages_par_m2_ef',
    'conso_chauffage_ef', 'conso_ecs_ef', 'type_batiment',
    'surface_habitable_logement', 'annee_construction',
    'date_etablissement_dpe', 'zone_climatique'
]

# Load the data (comment out if not executing)
# dpe = pd.read_csv('dpe03existant.csv', usecols=dpe_columns, low_memory=False)

# Convert date column
# dpe['date_etablissement_dpe'] = pd.to_datetime(dpe['date_etablissement_dpe'], errors='coerce')

# Filter: Keep DPEs from 2021 to 2024 (matching ENEDIS data range)
# dpe = dpe[(dpe['date_etablissement_dpe'].dt.year >= 2021) & 
#           (dpe['date_etablissement_dpe'].dt.year <= 2024)]

# Filter: Keep only valid DPE classes (A-G)
# valid_classes = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
# dpe = dpe[dpe['etiquette_dpe'].isin(valid_classes)]

# Filter: Remove unreasonable values
# dpe = dpe[dpe['surface_habitable_logement'] > 5] #eliminates unrealistic values
# dpe = dpe[dpe['conso_5_usages_par_m2_ef'].between(20, 600)] #Eliminates outliers 

# print(f"DPE data loaded: {len(dpe):,} rows (2021-2024)")

## 3. Load ENEDIS Electricity Consumption Data

ENEDIS data comes from the electricity distribution network operator. It contains **real measured consumption** for residential addresses.

### Required ENEDIS Columns

| Column | Description |
|--------|-------------|
| `Adresse` | Raw address string |
| `Code Commune` | INSEE code (to match with BAN) |
| `Nom Commune` | Municipality name |
| `Année` | Consumption year |
| `consommation_annuelle_moyenne_par_site_de_ladresse_mwh` | Average consumption per site (MWh) |
| `Nombre de logements` | Number of dwellings at this address |
| `Segment de client` | Customer segment (RESIDENTIEL only) |

In [7]:
# Load ENEDIS data

# enedis = pd.read_csv('consommation-annuelle-residentielle-par-adresse.csv', sep=';', low_memory=False)

# Filter: Keep 2021-2024 (matching DPE date range)
# enedis = enedis[(enedis['Année'] >= 2021) & (enedis['Année'] <= 2024)]

# Filter: Keep only residential customers
# enedis = enedis[enedis['Segment de client'] == 'RESIDENTIEL']

# Select relevant columns
# enedis_cols = [
#     'Adresse', 'Code Commune', 'Nom Commune', 'Année',
#     'consommation_annuelle_moyenne_par_site_de_ladresse_mwh',
#     'Nombre de logements', 'Segment de client'
# ]
# enedis = enedis[enedis_cols]

# print(f" ENEDIS data loaded: {len(enedis):,} rows (2021-2024)")

## 4. Create Standardized Join Key (Using BAN Logic)

### Strategy
We create an identical key in both datasets using the same logic:

**Key format**: `CLEANED_ADDRESS + '|' + INSEE_CODE + '|' + MUNICIPALITY_NAME`

### For DPE (already has BAN columns):
- Use `adresse_ban` (remove postal code if present)
- Use `code_insee_ban`
- Use `nom_commune_ban` (convert to uppercase)

### For ENEDIS (no BAN columns):
- Apply the same normalization manually:
  - Convert to uppercase
  - Strip whitespace
  - Combine `Adresse + Code Commune + Nom Commune`

In [8]:
# Create standardized join keys using BAN logic

# Remove postal code from adresse_ban (if present)
# dpe['adresse_clean'] = dpe['adresse_ban'].str.replace(r'\s*\d{5}.*$', '', regex=True).str.strip().str.upper()

# Create join key
# dpe['join_key'] = (
#     dpe['adresse_clean'].fillna('') + '|' +
#     dpe['code_insee_ban'].fillna('').astype(str) + '|' +
#     dpe['nom_commune_ban'].fillna('').str.upper()
# )

# print(f"DPE join keys created: {dpe['join_key'].nunique():,} unique keys")

# ----- ENEDIS: Normalize address and create key (same logic as BAN) -----

# enedis['adresse_clean'] = enedis['Adresse'].fillna('').str.upper().str.strip()
# enedis['Code Commune'] = enedis['Code Commune'].astype(str)

# Create join key
# enedis['join_key'] = (
#     enedis['adresse_clean'] + '|' +
#     enedis['Code Commune'] + '|' +
#     enedis['Nom Commune'].fillna('').str.upper()
# )

# print(f"ENEDIS join keys created: {enedis['join_key'].nunique():,} unique keys")

## 5. Deduplication - Keep Most Recent Record per Address

### Why deduplicate?
- A building may have multiple DPEs (different apartments, different dates)
- ENEDIS may have multiple years of consumption data

### Strategy
- **DPE**: Keep only the most recent DPE per address (by `date_etablissement_dpe`)
- **ENEDIS**: Keep only the most recent year per address (by `Année`)

### Important note
Because BAN removes apartment numbers, all apartments in the same building share the same `join_key`. This means we lose the ability to distinguish individual units. This is an accepted limitation for aggregate analysis.

In [9]:
# Keep most recent record per address

# ----- DPE: Keep most recent DPE per address -----
# dpe = dpe.sort_values(['join_key', 'date_etablissement_dpe'])
# dpe = dpe.groupby('join_key').tail(1).reset_index(drop=True)
# print(f"DPE after dedup: {len(dpe):,} records")

# ----- ENEDIS: Keep most recent year per address -----
# enedis = enedis.sort_values(['join_key', 'Année'])
# enedis = enedis.groupby('join_key').tail(1).reset_index(drop=True)
# print(f"ENEDIS after dedup: {len(enedis):,} records")

## 6. Merge DPE and ENEDIS

We perform an **INNER JOIN** on the `join_key`. This keeps only addresses that exist in BOTH datasets.

### What this means
- We lose addresses that only appear in one dataset
- The matched sample is not nationally representative
- However, it's sufficient for comparing DPE estimates against real consumption

### Expected match rate
Based on the reference analysis: ~4-5% of ENEDIS records match with DPE
- Reason: DPE covers all energy types (gas, oil, electricity); ENEDIS only electricity
- Address normalization differences also reduce matches

In [10]:
# Merge datasets

# Perform inner join
# merged = pd.merge(dpe, enedis, on='join_key', how='inner')

# print(f"\nMERGE COMPLETE")
# print(f"   Matched records: {len(merged):,}")
# print(f"   Columns in merged dataset: {len(merged.columns)}")
# print(f"   Match rate: {len(merged)/len(enedis)*100:.2f}% of ENEDIS records")

# Save merged dataset (compressed to save space)
# merged.to_csv('MERGED_DPE_ENEDIS.csv', index=False)
# print(f"\nSaved: MERGED_DPE_ENEDIS.csv")